Mechanism Hypotheses:
1. Lateral exchange w/ surrounding water. For a cyclone moving south, water with low CHL enters and dilutes/reduces CHL concentration. 
2. Eddy-wind interaction. Ekman pumping drives upwelling in anticyclones and downwelling in cyclones. But if this is the case, then we should expect a change/response at the centers first.
3. Seasonal changes and mixed layer depth changes. 

In [ ]:
from pathlib import Path
from typing import cast
import sys
import warnings

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator
from scipy.stats import binomtest, wilcoxon
from statsmodels.tools.sm_exceptions import ConvergenceWarning, SingularMatrixWarning

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from eddy_tracking.preprocess.tracks import PET_EPOCH
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_BOOTSTRAP = 2000
N_AGE_BINS = 5
RANDOM_SEED = 2026
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
polarity_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
identity_columns = ['polarity', 'track_id']

movement = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
plankton = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')
target_class = cast(pd.Series, movement['polarity']).map(target_classes)
movement['is_target'] = (
    movement['movement'].eq(target_class)
    | (
        movement['birth_distance_km'].abs().le(NEAR_AXIS_KM)
        & movement['death_side'].eq(target_class.str[1])
    )
)
analysis = cast(pd.DataFrame, plankton.merge(
    movement[identity_columns + ['is_target']], on=identity_columns, how='left',
).loc[lambda frame: frame['is_target']]).copy()
analysis['date'] = pd.to_datetime(analysis['date'])
analysis['eddy_key'] = analysis['polarity'] + ':' + analysis['track_id'].astype(str)
analysis['is_anticyclone'] = analysis['polarity'].eq('anticyclone').astype(int)
annual_angle = 2 * np.pi * (analysis['date'].dt.dayofyear - 1) / 365.25
analysis['season_sin'] = np.sin(annual_angle)
analysis['season_cos'] = np.cos(annual_angle)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

Are the changes that we're seeing the result of a few eddies with large increases/decreases or a consistent behavior across many eddies?

In [ ]:
bin_centers = (np.arange(N_AGE_BINS) + 0.5) / N_AGE_BINS
analysis['age_bin'] = np.minimum((analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
eddy_bins = analysis.groupby(identity_columns + ['age_bin'])['CHL'].mean()
rng = np.random.default_rng(RANDOM_SEED)
drop_rows = []
exclusion_effects = {}
curve_limits = [np.inf, -np.inf]
effect_limits = [np.inf, -np.inf]
loo_fig, loo_axes = cast(tuple[Figure, np.ndarray], plt.subplots(
    2, 2, figsize=(6.69, 4.8), sharex=True, sharey='row', layout='constrained',  # pyright: ignore[reportArgumentType]
    gridspec_kw={'height_ratios': (1.7, 1)},
))
for panel, polarity in enumerate(polarity_names):
    curve_ax = cast(Axes, loo_axes[0, panel])
    effect_ax = cast(Axes, loo_axes[1, panel])
    color = polarity_colors[polarity]
    table = eddy_bins.loc[polarity].unstack('age_bin').reindex(columns=range(N_AGE_BINS))
    matrix = table.to_numpy(dtype=float)  # (n_eddies, n_bins)
    finite = np.isfinite(matrix)
    counts = finite.sum(axis=0)
    sums = np.nansum(matrix, axis=0)
    full_mean = sums / counts
    without_one = (sums - np.where(finite, matrix, 0.0)) / (counts - finite)  # (n_eddies, n_bins)
    exclusion_effect = np.where(finite, without_one - full_mean, np.nan)
    exclusion_effects[polarity] = pd.DataFrame(exclusion_effect, index=table.index)
    curve_limits = [min(curve_limits[0], without_one.min()), max(curve_limits[1], without_one.max())]
    effect_limits = [min(effect_limits[0], np.nanmin(exclusion_effect)), max(effect_limits[1], np.nanmax(exclusion_effect))]
    for row in without_one:
        curve_ax.plot(bin_centers, row, color='#b0b0b0', linewidth=0.5, alpha=0.7, zorder=1)
    curve_ax.plot(bin_centers, full_mean, '-o', color=color, markersize=4, markeredgecolor='white', markeredgewidth=0.6, zorder=3)
    curve_ax.set_title(f'$\\bf{{({"ab"[panel]})}}$ {polarity_labels[polarity]} (n = {len(table)})', loc='left')
    curve_ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    curve_ax.set_axisbelow(True)
    effect_ax.axhline(0, color='#999999', linewidth=0.6, zorder=0)
    for age_bin in range(N_AGE_BINS):
        values = exclusion_effect[:, age_bin]
        values = values[np.isfinite(values)]
        jitter = rng.uniform(-0.04, 0.04, len(values))
        effect_ax.scatter(bin_centers[age_bin] + jitter, values, color=color, s=9, alpha=0.45, linewidths=0, zorder=2)
    effect_ax.set_title(f'$\\bf{{({"cd"[panel]})}}$ Effect of being excluded on bin mean', loc='left')
    effect_ax.set_xlabel('Fraction of observed track')
    effect_ax.set_xlim(0, 1)
    effect_ax.xaxis.set_ticks(np.linspace(0, 1, N_AGE_BINS + 1))
    effect_ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    effect_ax.set_axisbelow(True)
    full_difference = full_mean[-1] - full_mean[0]
    difference_effect = (without_one[:, -1] - without_one[:, 0]) - full_difference
    order = np.argsort(difference_effect * np.sign(full_difference))
    drop_row = {'polarity': polarity, 'largest_exclusion_effect': np.nanmax(np.abs(exclusion_effect))}
    for n_dropped in range(4):
        remaining = np.delete(matrix, order[:n_dropped], axis=0)
        drop_row[f'late_minus_early_dropping_{n_dropped}'] = np.nanmean(remaining[:, -1]) - np.nanmean(remaining[:, 0])
    drop_rows.append(drop_row)
for row_axes, low_high in ((loo_axes[0], curve_limits), (loo_axes[1], effect_limits)):
    ticks = cast(np.ndarray, MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]).tick_values(*low_high))
    for ax in row_axes:
        cast(Axes, ax).yaxis.set_ticks(ticks)
        cast(Axes, ax).set_ylim(ticks[0], ticks[-1])
loo_axes[0, 0].set_ylabel('Interior CHL (mg m$^{-3}$)')
loo_axes[1, 0].set_ylabel('Change in bin mean (mg m$^{-3}$)')
plt.show()
drop_table = pd.DataFrame(drop_rows).set_index('polarity')
display(drop_table.round(3))

One anticyclone in panel (d) pulls a bin mean by 0.05 mg m$^{-3}$, several times any other eddy. Which eddy is it, where did it go, and what does the target anticyclone curve look like without it?

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from eddy_tracking.config import load_config, resolve_data_dir
from eddy_tracking.preprocess.streamline import trace_mean_streamline
from eddy_tracking.preprocess.tracks import load_track_observations

CONTOUR_EVERY_DAYS = 5
cfg = load_config(EXPERIMENT)
outlier_id = int(exclusion_effects['anticyclone'].abs().max(axis=1).idxmax())
track_observations = load_track_observations(EXPERIMENT)
track = cast(pd.DataFrame, track_observations.loc[track_observations['polarity'].eq('anticyclone') & track_observations['track_id'].eq(outlier_id)]).sort_values('date')
swot_files = sorted(resolve_data_dir(cfg, 'swot_dir').glob('*.nc'))
mean_axis = trace_mean_streamline(swot_files, tuple(cfg['gulf_stream']['adt_level_range']))
contour_lon = np.concatenate(track['contour_lon'].to_list())
contour_lat = np.concatenate(track['contour_lat'].to_list())
lon_range, lat_range = cfg['base']['region']['lon_range'], cfg['base']['region']['lat_range']
extent = (max(np.floor(contour_lon.min()) - 1, lon_range[0]), min(np.ceil(contour_lon.max()) + 1, lon_range[1]), max(np.floor(contour_lat.min()) - 1, lat_range[0]), min(np.ceil(contour_lat.max()) + 1, lat_range[1]))
age_days = (track['date'] - track['date'].iloc[0]).dt.days
age_norm = Normalize(0, int(age_days.iloc[-1]))
age_cmap = ListedColormap(plt.get_cmap('Reds')(np.linspace(0.3, 1, 256)))
map_height = 3.35 * 0.84 * (extent[3] - extent[2]) / (extent[1] - extent[0])

fig = plt.figure(figsize=(3.35, map_height + 0.95))
ax = cast(GeoAxes, fig.add_subplot(fig.add_gridspec(1, 1, left=0.14, right=0.98, bottom=0.7 / (map_height + 0.95), top=1 - 0.25 / (map_height + 0.95))[0], projection=ccrs.PlateCarree()))
ax.set_extent(extent, crs=ccrs.PlateCarree())
ax.plot(mean_axis.lon, mean_axis.lat, color='black', linewidth=1.0, linestyle='--', transform=ccrs.PlateCarree(), zorder=4)
drawn = track.loc[age_days.mod(CONTOUR_EVERY_DAYS).eq(0) | track['date'].eq(track['date'].iloc[-1])]
for ring_lon, ring_lat, age in zip(drawn['contour_lon'], drawn['contour_lat'], age_days.loc[drawn.index]):
    ax.plot(np.append(ring_lon, ring_lon[0]), np.append(ring_lat, ring_lat[0]), color=age_cmap(age_norm(age)), linewidth=0.7, transform=ccrs.PlateCarree(), zorder=5)
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=2)
ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', xlocs=range(-80, -50, 2), ylocs=range(30, 50, 1), zorder=3)
gridlines.top_labels = False
gridlines.right_labels = False
gridlines.xlabel_style = {'size': 8}
gridlines.ylabel_style = {'size': 8}
ax.set_title(f'Anticyclone {outlier_id}, speed contour every {CONTOUR_EVERY_DAYS} days', loc='left')
ax.legend(handles=[Line2D([], [], color='black', linewidth=1.0, linestyle='--', label='Mean Gulf Stream axis')], loc='upper left', frameon=True, framealpha=0.95, edgecolor='none', handlelength=2.2, borderpad=0.3)
age_bar = fig.colorbar(ScalarMappable(norm=age_norm, cmap=age_cmap), cax=ax.inset_axes((0, -0.32 / map_height, 1, 0.08 / map_height)), orientation='horizontal')
age_bar.set_label('Days since first detection')
age_bar.ax.tick_params(length=2)
age_bar.outline.set_linewidth(0.6)
plt.show()

In [ ]:
def summarize_bins(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    matrix = cast(pd.DataFrame, frame.groupby(['track_id', 'age_bin'])[column].mean().unstack('age_bin')).reindex(columns=range(N_AGE_BINS)).to_numpy(dtype=float)
    counts = np.isfinite(matrix).sum(axis=0)
    sampled = matrix[rng.integers(0, len(matrix), size=(N_BOOTSTRAP, len(matrix)))]
    sampled_counts = np.isfinite(sampled).sum(axis=1)
    sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_counts, out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0)
    low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
    return pd.DataFrame({'age_midpoint': bin_centers, 'mean': np.nansum(matrix, axis=0) / counts, 'ci_low': low, 'ci_high': high, 'n_eddies': counts})


rng = np.random.default_rng(RANDOM_SEED)
anticyclones = cast(pd.DataFrame, analysis.loc[analysis['polarity'].eq('anticyclone')]).sort_values(['track_id', 'date']).copy()
anticyclones['change'] = anticyclones['CHL'] - anticyclones.groupby('track_id')['CHL'].transform('first')
cohorts = {
    f'All target anticyclones (n = {anticyclones["track_id"].nunique()})': anticyclones,
    f'Without anticyclone {outlier_id} (n = {anticyclones["track_id"].nunique() - 1})': anticyclones.loc[anticyclones['track_id'].ne(outlier_id)],
}
cohort_colors = dict(zip(cohorts, ('#b2182b', '#f4a582')))
summaries = {(label, column): summarize_bins(frame, column) for label, frame in cohorts.items() for column in ('change', 'CHL')}

fig, axes = cast(tuple[Figure, np.ndarray], plt.subplots(1, 2, figsize=(6.69, 2.6), sharex=True, layout='constrained'))
for ax, letter, column, title in zip(axes, 'ab', ('change', 'CHL'), ('Change in CHL from first observation', 'Interior CHL')):
    ax = cast(Axes, ax)
    if column == 'change':
        ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    for offset, label in zip((-0.012, 0.012), cohorts):
        summary = summaries[label, column]
        ax.errorbar(summary['age_midpoint'] + offset, summary['mean'], yerr=[summary['mean'] - summary['ci_low'], summary['ci_high'] - summary['mean']], fmt='none', ecolor=cohort_colors[label], capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=2)
        ax.plot(summary['age_midpoint'] + offset, summary['mean'], '-o', color=cohort_colors[label], linewidth=1.2, markersize=3, markeredgecolor='white', markeredgewidth=0.5, label=label, zorder=3)
    ax.set_title(f'$\\bf{{({letter})}}$ {title}', loc='left')
    ax.set_xlabel('Fraction of observed track')
    ax.set_xlim(0, 1)
    ax.xaxis.set_ticks(np.linspace(0, 1, N_AGE_BINS + 1))
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ticks = cast(np.ndarray, MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]).tick_values(*ax.get_ylim()))
    ax.yaxis.set_ticks(ticks)
    ax.set_ylim(ticks[0], ticks[-1])
cast(Axes, axes[0]).set_ylabel('$\\Delta$CHL (mg m$^{-3}$)')
cast(Axes, axes[1]).set_ylabel('Interior CHL (mg m$^{-3}$)')
cast(Axes, axes[1]).legend(loc='lower right', handlelength=2.2, borderaxespad=0.2, labelspacing=0.3)
plt.show()
comparison = pd.concat({label: summary.set_index('age_midpoint')[['mean', 'ci_low', 'ci_high']] for (label, column), summary in summaries.items() if column == 'change'}, axis=1)
comparison['difference'] = comparison[(list(cohorts)[1], 'mean')] - comparison[(list(cohorts)[0], 'mean')]
display(comparison.round(3))

Figure 3 of `figures.ipynb`, the target cohort with the anticyclone above excluded.

In [ ]:
import xarray as xr

N_RADIAL_BINS = 10
MAX_RADIUS = 2
shown_pigments = {'Tchla': 'Total chlorophyll-a', 'Fuco': 'Fucoxanthin', 'HexFuco': "19'-Hex-fucoxanthin", 'Perid': 'Peridinin', 'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a'}
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)
pigment_table = pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet')
pigment_table['polarity'] = cast(pd.Series, pigment_table['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
plankton_files = sorted(resolve_data_dir(cfg, 'plankton_dir').glob('plankton_*.nc'))
eddies = movement.loc[movement['is_target'] & ~(movement['polarity'].eq('anticyclone') & movement['track_id'].eq(outlier_id)), identity_columns]
date_ranges = {}
for year in range(plankton['date'].min().year, plankton['date'].max().year + 1):
    start = pd.Timestamp(year, 1, 1)
    while start.year == year:
        end = min(start + pd.Timedelta(days=7), pd.Timestamp(year, 12, 31))
        date_ranges[start + pd.Timedelta(days=(end - start).days // 2)] = (start, end)
        start = end + pd.Timedelta(days=1)
with xr.open_dataset(plankton_files[0]) as ds:
    lon = ds['longitude'].to_numpy()
    lat = ds['latitude'].to_numpy()
chl_field = xr.open_mfdataset(plankton_files, combine='by_coords')['CHL']

cohort_plankton = plankton.merge(eddies, on=identity_columns)
cohort_pigments = pigment_table.merge(eddies, on=identity_columns)
lifetime = pd.concat([
    cohort_plankton[identity_columns + ['date', 'age_frac']].assign(source='chl', variable='CHL', concentration=cohort_plankton['CHL']),
    cohort_pigments[identity_columns + ['date', 'age_frac']].assign(source='sdp').join(
        cast(pd.DataFrame, cohort_pigments[[f'eddy_mean_{pigment}' for pigment in shown_pigments]]).rename(columns={f'eddy_mean_{pigment}': pigment for pigment in shown_pigments}),
    ).melt(id_vars=identity_columns + ['date', 'age_frac', 'source'], var_name='variable', value_name='concentration'),
], ignore_index=True).sort_values(identity_columns + ['variable', 'date'])
lifetime['age_bin'] = np.minimum((lifetime['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
lifetime['change'] = lifetime['concentration'] - lifetime.groupby(identity_columns + ['variable'])['concentration'].transform('first')
lifetime_bins = lifetime.groupby(['source', 'variable'] + identity_columns + ['age_bin'])['change'].mean().reset_index()
n_eddies = lifetime.groupby(['source', 'polarity'])['track_id'].nunique()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for keys, source_bins in lifetime_bins.groupby(['source', 'polarity']):
    source, polarity = cast(tuple[str, str], keys)
    eddy_ids = sorted(source_bins['track_id'].unique())
    draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
    for variable, variable_bins in source_bins.groupby('variable'):
        matrix = variable_bins.pivot(index='track_id', columns='age_bin', values='change').reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
        counts = np.isfinite(matrix).sum(axis=0)
        means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
        sampled = matrix[draws]
        sampled_counts = np.isfinite(sampled).sum(axis=1)
        sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_counts, out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0)
        low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
        summary_rows.append(pd.DataFrame({
            'source': source, 'polarity': polarity, 'variable': variable, 'age_bin': range(N_AGE_BINS),
            'age_midpoint': bin_centers, 'mean': means, 'ci_low': low, 'ci_high': high, 'n_eddies': counts,
        }))
lifetime_summary = pd.concat(summary_rows, ignore_index=True)

profiles = pd.merge_asof(
    cast(pd.DataFrame, lifetime.loc[lifetime['variable'].eq('CHL')]).sort_values('date'),
    cast(pd.DataFrame, track_observations[identity_columns + ['date', 'radius_km']]).sort_values('date'),
    on='date', by=identity_columns, direction='nearest',
).merge(cohort_plankton[identity_columns + ['date', 'center_lon', 'center_lat']], on=identity_columns + ['date'])
ring_chl = np.full((len(profiles), N_RADIAL_BINS), np.nan)
for date, rows in profiles.groupby('date'):
    composite = chl_field.sel(time=slice(*date_ranges[date])).mean('time').to_numpy()
    for index, center_lon, center_lat, radius_km in zip(rows.index, rows['center_lon'], rows['center_lat'], rows['radius_km']):
        half_width = MAX_RADIUS * radius_km / 111.0
        lat_idx = np.flatnonzero(np.abs(lat - center_lat) <= half_width)
        lon_idx = np.flatnonzero(np.abs(lon - center_lon) <= half_width / np.cos(np.radians(center_lat)))
        lon_grid, lat_grid = np.meshgrid(lon[lon_idx], lat[lat_idx])
        distance = np.hypot((lon_grid - center_lon) * np.cos(np.radians(center_lat)), lat_grid - center_lat) * 111.0 / radius_km
        values = composite[np.ix_(lat_idx, lon_idx)]
        valid = np.isfinite(values) & (distance < MAX_RADIUS)
        ring = np.digitize(distance[valid], radial_edges) - 1
        counts = np.bincount(ring, minlength=N_RADIAL_BINS)
        ring_chl[index] = np.divide(np.bincount(ring, weights=values[valid], minlength=N_RADIAL_BINS), counts, out=np.full(N_RADIAL_BINS, np.nan), where=counts > 0)
radial = profiles[identity_columns + ['age_bin']].join(pd.DataFrame(ring_chl, columns=range(N_RADIAL_BINS))).melt(id_vars=identity_columns + ['age_bin'], var_name='radial_bin', value_name='CHL')
radial_grid = radial.groupby(identity_columns + ['age_bin', 'radial_bin'])['CHL'].mean().groupby(['polarity', 'age_bin', 'radial_bin']).mean()
chl_norm = Normalize(np.floor(radial_grid.min() / 0.02) * 0.02, np.ceil(radial_grid.max() / 0.02) * 0.02)


def draw_change(ax: Axes, source: str, variable: str) -> None:
    ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    for polarity, color in polarity_colors.items():
        result = lifetime_summary.loc[lifetime_summary['polarity'].eq(polarity) & lifetime_summary['variable'].eq(variable)].sort_values('age_bin')
        ax.errorbar(result['age_midpoint'], result['mean'], yerr=[result['mean'] - result['ci_low'], result['ci_high'] - result['mean']], fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=2)
        ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, linewidth=1.2, markersize=3, markeredgecolor='white', markeredgewidth=0.5, label=f'{polarity_labels[polarity]} (n = {n_eddies[source, polarity]})', zorder=3)
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.set_xlim(0, 1)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ax.yaxis.set_major_locator(locator)
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    ax.yaxis.set_ticks(ticks)
    ax.set_ylim(ticks[0], ticks[-1])


fig = plt.figure(figsize=(6.69, 6.6))
rows = fig.add_gridspec(2, 1, left=0.1, right=0.9, bottom=0.11, top=0.955, height_ratios=(2.0, 4.0), hspace=0.32)
top = rows[0].subgridspec(1, 2, width_ratios=(1.6, 1.45), wspace=0.35)
chl_ax = cast(Axes, fig.add_subplot(top[0]))
heat_axes = cast(np.ndarray, top[1].subgridspec(1, 2, wspace=0.2).subplots(sharey=True))
pigment_axes = cast(np.ndarray, rows[1].subgridspec(2, 3, wspace=0.45, hspace=0.4).subplots(sharex=True))
draw_change(chl_ax, 'chl', 'CHL')
chl_ax.set_title('$\\bf{(a)}$ Change in CHL', loc='left', pad=12.4)
chl_ax.set_ylabel('$\\Delta$CHL (mg m$^{-3}$)')
chl_ax.set_xlabel('Fraction of observed track')
chl_ax.legend(loc='lower left', handlelength=2.2, borderaxespad=0.2, labelspacing=0.3)
heat_cmap = plt.get_cmap('viridis').copy()
heat_cmap.set_bad('#e6e6e6')
for ax, polarity in zip(heat_axes, polarity_colors):
    ax = cast(Axes, ax)
    grid = radial_grid.loc[polarity].unstack('radial_bin').reindex(index=range(N_AGE_BINS), columns=range(N_RADIAL_BINS)).to_numpy(dtype=float).T
    ax.pcolormesh(bin_edges, radial_edges, np.ma.masked_invalid(grid), cmap=heat_cmap, norm=chl_norm, edgecolors='white', linewidth=0.5)
    ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
    ax.set_aspect('equal')
    ax.set_title(f'{polarity.capitalize()}s', pad=2)
    ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'])
    ax.xaxis.set_ticks(bin_edges, minor=True)
    ax.yaxis.set_ticks([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1', '1.5', '2'])
    ax.yaxis.set_ticks(radial_edges, minor=True)
    ax.tick_params(length=2)
    ax.tick_params(which='minor', length=1.2)
cast(Axes, heat_axes[0]).set_ylabel('Distance from center (speed radii)')
cast(Axes, heat_axes[0]).text(0, 1.1, '$\\bf{(b)}$ CHL by radius and age', transform=heat_axes[0].transAxes, va='bottom', ha='left')
cast(Axes, heat_axes[0]).set_xlabel('Fraction of observed track')
cast(Axes, heat_axes[0]).xaxis.set_label_coords(1.1, -0.12)
heat_bar = fig.colorbar(ScalarMappable(norm=chl_norm, cmap=heat_cmap), cax=cast(Axes, heat_axes[1]).inset_axes((1.1, 0, 0.08, 1)))
heat_bar.set_label('CHL (mg m$^{-3}$)')
heat_bar.ax.tick_params(length=2)
heat_bar.ax.yaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]))
heat_bar.outline.set_linewidth(0.5)
for ax, letter, (pigment, label) in zip(pigment_axes.flat, 'cdefgh', shown_pigments.items()):
    ax = cast(Axes, ax)
    draw_change(ax, 'sdp', pigment)
    ax.set_title(f'$\\bf{{({letter})}}$ {label}', loc='left')
for ax in pigment_axes[-1]:
    cast(Axes, ax).set_xlabel('Fraction of observed track')
pigment_box = rows[1].get_position(fig)
fig.text(0.008, (pigment_box.y0 + pigment_box.y1) / 2, 'Change from first observation (mg m$^{-3}$)', rotation=90, va='center', ha='left')
fig.legend(*cast(Axes, pigment_axes.flat[0]).get_legend_handles_labels(), loc='upper center', bbox_to_anchor=(0.5, pigment_box.y0 - 0.07), ncol=2)
plt.show()